# 1. Imports

In [ ]:
import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

import numpy as np 
import matplotlib.pyplot as plt
from tqdm import tqdm  
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import pandas as pd
import re

# --- Custom imports ---
from core.helpers import load_dataset_paths, compute_average_signal, load_preprocessed_signal, get_data, load_blind_signal
from core import config
from classifiers.CNN_classifier import CNN1D
from core.preprocessing import preprocess_signal, preprocess_save_all, extract_PulseWidth_SignalPower
import warnings
warnings.filterwarnings("ignore", message="The structure of `inputs` doesn't match the expected structure")
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

# 2. Loading Preprocessed Files

In [ ]:
DEVICES = config.DEVICES
LABELS = config.LABELS
TMAX = config.TMAX
LR = config.LR
EPOCHS = config.EPOCHS
BATCHSIZE = config.BATCHSIZE
DROPOUT = config.DROPOUT
L2_LAMBDA = config.L2_LAMBDA
LEVEL = 4

preprocess_save_all(
    normalize=True,
    tmax=TMAX,
    SNR_filtering=True,
    do_artifact_removal=False,
    include_blind=True,
    do_dwt_downsampling=True
)

all_paths_preprocessed = load_dataset_paths()
blind_files_sorted = load_blind_signal()

for device in DEVICES:
    print(f"\n{device}:")
    for label in LABELS:
        print(f"{label}: {len(all_paths_preprocessed[device][label])}")

print(f"\nBLIND: {len(blind_files_sorted)}")

# 3. Classification

In [ ]:
X, labels, _, _ = get_data(device="PRIMA_LE_DA", classes=2)

cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)

y_true, y_pred, model, ig = cnn.fit_train_val(
    X, labels, epochs=EPOCHS, batch_size=BATCHSIZE, val_size=0.2, compute_ig=True)
results = cnn.evaluate(y_true, y_pred, verbose=False)

print("Metrics on PRIMA_LE_DA BC Only vs BC + RGC:")
print(f"F1 Score: {results['f1']:.4f}")

# 4. Integrated Gradients Analysis

In [ ]:
CONDITION_COLORS = {
    "BC_Only": "grey",
    "BC_and_RGC": "black",
}

LABEL_FONTSIZE = 15

DEVICE_TITLES = {
    "PRIMA_LE_DA": "PRIMA LE DA",
    "PRIMA_RCS_DA": "PRIMA RCS DA",
    "MP20_LE_DA": "MP20 LE DA",
    "MP20_RCS_LA": "MP20 RCS LA",
}

#TODO: parse_filename_info and select_irradiance_samples are used in both this notebook and data_visualization.ipynb - consider moving to helpers.py

def parse_filename_info(filename):
    parts = os.path.splitext(os.path.basename(filename))[0].split("_")
    device = parts[0] if parts[0] not in ("LE", "ML") else (parts[0] if parts[1].isdigit() else parts[1])
    try:
        animal = int(parts[2] if parts[0] == "RCS" else (parts[1] if parts[1].isdigit() else parts[2]))
    except (ValueError, IndexError):
        animal = None
    return animal, device


def select_irradiance_samples(files, n):
    if not n or n <= 0 or n >= len(files):
        return files
    idxs = np.unique(np.round(np.linspace(0, len(files) - 1, n)).astype(int))
    if len(idxs) < n:
        candidates = [i for i in range(len(files)) if i not in set(idxs)]
        extra = np.round(np.linspace(0, len(candidates) - 1, n - len(idxs))).astype(int)
        idxs = np.sort(np.concatenate([idxs, [candidates[i] for i in extra]]))
    return [files[i] for i in idxs[:n]]


def get_preprocessed_path(original_path):
    parts = original_path.split(os.sep)
    parts[1] = "Preprocessed_VEP_Data"
    return os.sep.join(parts)

def iter_preprocessed_files(base_path, device_folder, conditions=("BC_Only", "BC_and_RGC")):
    for condition in conditions:
        condition_path = os.path.join(base_path, device_folder, condition)
        if not os.path.isdir(condition_path):
            continue
        for entry in os.scandir(condition_path):
            if entry.is_file() and entry.name.endswith(".csv"):
                animal, device = parse_filename_info(entry.name)
                if animal is not None:
                    orig_path = entry.path
                    prep_path = get_preprocessed_path(orig_path)
                    if os.path.exists(prep_path):
                        pw, irr = extract_PulseWidth_SignalPower(orig_path)
                        yield {"filepath": prep_path, "original_path": orig_path, "animal_number": animal, 
                               "pulse_width": pw, "irradiance": irr, "condition": condition, "device": device}


def compute_ig_for_signal(model, cnn, signal, condition):
    label_to_index = {v: k for k, v in cnn.label_decoder.items()}
    target_class = label_to_index.get(condition, 0)
    ig = cnn.integrated_gradients(model=model, x=signal, target_class=target_class, steps=50)
    return ig

def plot_animal_traces_with_ig(base_path, device_folder, model, cnn, target_pulse_width=10.0,
                                tolerance=0.01, n_traces=None, vline_x=30.0):
    per_animal = {}
    for rec in iter_preprocessed_files(base_path, device_folder):
        if abs(rec["pulse_width"] - target_pulse_width) < tolerance:
            per_animal.setdefault(rec["animal_number"], {}).setdefault(rec["irradiance"], rec)

    if not per_animal:
        return

    irr_map = max(per_animal.values(), key=len)
    files = select_irradiance_samples([irr_map[k] for k in sorted(irr_map)], n_traces)
    n = len(files)
    if not n:
        return

    title = DEVICE_TITLES.get(device_folder, device_folder)
    fig, axes = plt.subplots(n, 1, figsize=(7, n * 1.1), squeeze=False)
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)

    unit_label = fig.text(0, 0, "mW/mm²", fontsize=LABEL_FONTSIZE, ha="center", va="bottom")

    # First pass: determine common x-limits
    t_min, t_max = None, None
    for rec in files:
        t, _ = load_preprocessed_signal(rec["filepath"])
        t_min = t[0] if t_min is None else min(t_min, t[0])
        t_max = t[-1] if t_max is None else max(t_max, t[-1])

    for i, rec in enumerate(files):
        t, s = load_preprocessed_signal(rec["filepath"])

        ig = compute_ig_for_signal(model, cnn, s, rec["condition"])
        ig_absmax = np.max(np.abs(ig)) + 1e-12
        ig_norm = ig / ig_absmax

        ax = axes[i, 0]

        extent = [t[0], t[-1], s.min(), s.max()]
        ax.imshow(ig_norm[np.newaxis, :], aspect="auto", cmap="plasma", alpha=0.5,
                  extent=extent, origin="lower", vmin=-1, vmax=1, zorder=1)

        trace_color = "dimgrey" if rec["condition"] == "BC_Only" else "black"
        ax.plot(t, s, color=trace_color, linewidth=2.5, zorder=2)

        # Y-label: only the irradiance number (unit is in header)
        ax.set_ylabel(f"{rec['irradiance']:.2f}", rotation=0, labelpad=35,
                      va="center", fontsize=LABEL_FONTSIZE)

        yr = max(s.max() - s.min(), 0.1)
        ax.set_ylim(s.min() - 0.15 * yr, s.max() + 0.15 * yr)
        ax.set_xlim(t_min, t_max)

        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])

    # Legend for conditions
    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color="dimgrey", lw=2.5, label="BC"),
        Line2D([0], [0], color="black", lw=2.5, label="BC + RGC"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", fontsize=LABEL_FONTSIZE,
               frameon=False, bbox_to_anchor=(1.0, 1.0))

    plt.tight_layout(rect=[0.05, 0, 1, 0.99])
    plt.subplots_adjust(hspace=0.15)

    # Center "mW/mm²" header over the y-axis labels
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig_width = fig.get_window_extent(renderer).width
    bbox = axes[0, 0].yaxis.label.get_window_extent(renderer)
    label_x_norm = (bbox.x0 + bbox.width / 2) / fig_width
    top_y = axes[0, 0].get_position().y1
    unit_label.set_position((label_x_norm, top_y + 0.01))
    unit_label.set_transform(fig.transFigure)

    plt.show()

X, labels, _, _ = get_data(device="PRIMA_LE_DA", classes=2)
cnn = CNN1D(regularization="l2", learning_rate=LR, l2_lambda=L2_LAMBDA, dropout_rate=DROPOUT)
y_true, y_pred, model, ig = cnn.fit_train_val(X, labels, epochs=EPOCHS, batch_size=BATCHSIZE, val_size=0.2, compute_ig=True)

base_path = "data/Labelled_VEP_Data"
plot_animal_traces_with_ig(base_path, "PRIMA_LE_DA", model, cnn, target_pulse_width=10.0, n_traces=10, vline_x=15.0)
plot_animal_traces_with_ig(base_path, "PRIMA_RCS_DA", model, cnn, target_pulse_width=10.0, n_traces=10, vline_x=15.0)   
plot_animal_traces_with_ig(base_path, "MP20_LE_DA", model, cnn, target_pulse_width=10.0, n_traces=10, vline_x=15.0)
plot_animal_traces_with_ig(base_path, "MP20_RCS_LA", model, cnn, target_pulse_width=10.0, n_traces=0, vline_x=15.0)
plot_animal_traces_with_ig(base_path, "RB20_RCS_LA", model, cnn, target_pulse_width=10.0, n_traces=10, vline_x=15.0)

In [ ]:
# ── make sure these are imported / defined before this cell ──────────────────
from scipy.signal import find_peaks
from matplotlib.lines import Line2D
import numpy as np, os

# ── peak helpers ─────────────────────────────────────────────────────────────

def find_peaks_and_valleys_1d(signal,
                               pos_prominence_pct=15, neg_prominence_pct=15,
                               distance=3, pos_after_first_neg=False,
                               max_pos_peaks=None, max_neg_peaks=None):
    signal = np.asarray(signal)
    signal_range = np.max(signal) - np.min(signal)
    if signal_range <= 1e-12:
        return np.array([], dtype=int), np.array([], dtype=int)

    pos_prominence = signal_range * (pos_prominence_pct / 100.0)
    neg_prominence = signal_range * (neg_prominence_pct / 100.0)

    neg_peaks, _ = find_peaks(-signal, prominence=neg_prominence, distance=distance)
    pos_peaks, _ = find_peaks( signal, prominence=pos_prominence, distance=distance)

    neg_peaks = neg_peaks[np.argsort(neg_peaks)]
    pos_peaks = pos_peaks[np.argsort(pos_peaks)]

    if pos_after_first_neg and len(neg_peaks) > 0:
        pos_peaks = pos_peaks[pos_peaks > neg_peaks[0]]
    if max_pos_peaks is not None:
        pos_peaks = pos_peaks[:max_pos_peaks]
    if max_neg_peaks is not None:
        neg_peaks = neg_peaks[:max_neg_peaks]

    return pos_peaks, neg_peaks


def get_peak_params_for_condition(condition):
    if condition == "BC_Only":
        return dict(pos_prominence_pct=15, neg_prominence_pct=15,
                    distance=3, pos_after_first_neg=True,
                    max_pos_peaks=3, max_neg_peaks=3)
    elif condition == "BC_and_RGC":
        return dict(pos_prominence_pct=10, neg_prominence_pct=15,
                    distance=3, pos_after_first_neg=True,
                    max_pos_peaks=3, max_neg_peaks=3)
    else:
        return dict(pos_prominence_pct=15, neg_prominence_pct=15,
                    distance=3, pos_after_first_neg=True,
                    max_pos_peaks=3, max_neg_peaks=3)


def annotate_peaks(ax, x, signal, condition):
    params = get_peak_params_for_condition(condition)
    pos_peaks, neg_peaks = find_peaks_and_valleys_1d(signal, **params)
    yr = max(signal.max() - signal.min(), 0.1)
    xr = x[-1] - x[0]

    for i, idx in enumerate(pos_peaks, 1):
        ax.plot(x[idx], signal[idx], 'o', color='black', markersize=4, zorder=4)
        ax.text(x[idx] + 0.01 * xr, signal[idx] + 0.01 * yr, f'P{i}',  # ← right + up
                fontsize=LABEL_FONTSIZE-2, fontweight='bold',
                ha='left', va='bottom', color='black', zorder=5)

    for i, idx in enumerate(neg_peaks, 1):
        ax.plot(x[idx], signal[idx], 'o', color='black', markersize=4, zorder=4)
        ax.text(x[idx] + 0.01 * xr, signal[idx] - 0.01 * yr, f'N{i}',  # ← right + down
                fontsize=LABEL_FONTSIZE-2, fontweight='bold',
                ha='left', va='top', color='black', zorder=5)


# ── main grid function ────────────────────────────────────────────────────────

def plot_2x2_ig_grid(base_path, device_configs, model, cnn,
                     target_pulse_width=10.0, tolerance=0.01,
                     show_original=False, crop_dwt_start_pct=0.05,
                     annotate_first_panel=True,               # ← NEW
                     save_path="outputs/figures/IG_example"):
    """
    2x2 grid of IG panels.
    Only the first panel (PRIMA LE DA, top-left) gets P/N annotations
    when annotate_first_panel=True.
    """
    grid_positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

    # ── data collection ───────────────────────────────────────────────────────
    all_data = []
    for cfg in device_configs:
        device_folder   = cfg["device_folder"]
        irradiance_list = cfg["irradiance_list"]

        per_animal = {}
        for rec in iter_preprocessed_files(base_path, device_folder):
            if abs(rec["pulse_width"] - target_pulse_width) < tolerance:
                per_animal.setdefault(rec["animal_number"], {}).setdefault(rec["irradiance"], rec)

        if not per_animal:
            all_data.append([])
            continue

        irr_map              = max(per_animal.values(), key=len)
        available_irradiances = sorted(irr_map.keys())
        files = []

        for target_irr in irradiance_list:
            closest_irr = min(available_irradiances, key=lambda x: abs(x - target_irr))
            if abs(closest_irr - target_irr) / target_irr < 0.05 and closest_irr in irr_map:
                files.append(irr_map[closest_irr])

        preprocessed = []
        t_min, t_max = None, None

        for rec in files:
            t_dwt_full, s_dwt_full = load_preprocessed_signal(rec["filepath"])
            mask = t_dwt_full <= TMAX
            t_dwt_full, s_dwt_full = t_dwt_full[mask], s_dwt_full[mask]
            crop_idx  = int(len(s_dwt_full) * crop_dwt_start_pct)
            s_dwt     = s_dwt_full[crop_idx:]
            t_dwt = np.linspace(0, TMAX, len(s_dwt))

            t_orig, s_orig = (load_signal_no_dwt(rec["original_path"],
                                                  normalize=True, tmax=TMAX)
                              if show_original else (None, None))

            ig         = compute_ig_for_signal(model, cnn, s_dwt_full, rec["condition"])
            ig_cropped = ig[crop_idx:]
            ig_norm    = ig_cropped / (np.max(np.abs(ig_cropped)) + 1e-12)

            t_min = t_dwt[0]  if t_min is None else min(t_min, t_dwt[0])
            t_max = t_dwt[-1] if t_max is None else max(t_max, t_dwt[-1])

            preprocessed.append((t_dwt, s_dwt, ig_norm, t_orig, s_orig, rec))

        all_data.append((preprocessed, t_min, t_max))

    # ── figure layout ─────────────────────────────────────────────────────────
    n_traces   = max(len(d[0]) for d in all_data if d)
    fig        = plt.figure(figsize=(14, n_traces * 2.5))
    outer_grid = fig.add_gridspec(2, 2, hspace=0.18, wspace=0.25)

    for panel_idx, (cfg, data) in enumerate(zip(device_configs, all_data)):
        row, col = grid_positions[panel_idx]
        title    = DEVICE_TITLES.get(cfg["device_folder"], cfg["device_folder"])

        if not data:
            ax = fig.add_subplot(outer_grid[row, col])
            ax.axis("off")
            continue

        preprocessed, t_min, t_max = data
        n          = len(preprocessed)
        #inner_grid = outer_grid[row, col].subgridspec(n, 1, hspace=0.0)
        inner_grid = outer_grid[row, col].subgridspec(n, 1, hspace=0.15)

        for i, (t_dwt, s_dwt, ig_norm, t_orig, s_orig, rec) in enumerate(preprocessed):
            ax = fig.add_subplot(inner_grid[i])

            # IG heatmap
            extent = [t_dwt[0], t_dwt[-1], s_dwt.min(), s_dwt.max()]
            ax.imshow(ig_norm[np.newaxis, :], aspect="auto", cmap="plasma", alpha=0.5,
                      extent=extent, origin="lower", vmin=-1, vmax=1, zorder=1)

            # Optional original signal underlay
            if show_original and t_orig is not None:
                ax.plot(t_orig, s_orig, color="lightgray", linewidth=1.5, alpha=0.6, zorder=2)

            # DWT trace
            trace_color = "dimgrey" if rec["condition"] == "BC_Only" else "black"
            ax.plot(t_dwt, s_dwt, color=trace_color, linewidth=2.5, zorder=3)

            # ── P/N annotations — first panel only ───────────────────────────
            if annotate_first_panel and panel_idx == 0:
                annotate_peaks(ax, t_dwt, s_dwt, rec["condition"])

            # Irradiance y-label
            ax.set_ylabel(f"{rec['irradiance']:.2f}", rotation=0, labelpad=30,
                          va="center", fontsize=LABEL_FONTSIZE - 1)

            yr = max(s_dwt.max() - s_dwt.min(), 0.1)
            ax.set_ylim(s_dwt.min() - 0.05 * yr, s_dwt.max() + 0.05 * yr)
            ax.set_xlim(t_min, t_max)

            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.set_yticks([])
            ax.set_xticks([])

            if annotate_first_panel and panel_idx == 0:
                annotate_peaks(ax, t_dwt, s_dwt, rec["condition"])
                ax.set_zorder(10)        # ← this axis renders on top of all others
                ax.patch.set_visible(True)   # ← make the axes background opaque, blocking the heatmap below
                ax.patch.set_facecolor("white")  # or whatever your figure background is
                ax.patch.set_alpha(0.0)  # keep it transparent normally...

            # Top trace decorations
            if i == 0:
                ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
                if row == 0:                           # unit label only on top row
                    ax.text(-0.12, 1.25, "mW/mm²",
                            transform=ax.transAxes,
                            fontsize=LABEL_FONTSIZE, ha="center", va="bottom")
                if panel_idx == 1:                     # legend in top-right panel
                    legend_handles = [
                        Line2D([0], [0], color="dimgrey", lw=3, label="BC"),
                        Line2D([0], [0], color="black",   lw=3, label="BC + RGC"),
                    ]
                    ax.legend(handles=legend_handles, fontsize=LABEL_FONTSIZE,
                              frameon=False, loc="lower left",
                              bbox_to_anchor=(0.7, 1.05), ncol=1,
                              borderaxespad=0, handlelength=1.2)

            # Bottom trace: time axis only on bottom row
            # if i == n - 1 and row == 1:
            #     tick_positions = np.arange(0, TMAX + 1, 100)
            #     ax.spines['bottom'].set_visible(True)
            #     ax.spines['bottom'].set_position(('outward', 4))
            #     ax.set_xticks(tick_positions)
            #     ax.set_xticklabels([str(int(x)) for x in tick_positions], fontsize=LABEL_FONTSIZE - 1)
            #     ax.set_xlabel("Time (ms)", fontsize=LABEL_FONTSIZE - 1)
            #     ax.set_xlim(0, TMAX)   # ← set xlim AFTER xticks so it isn't overridden

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(f"{save_path}.png", dpi=300, bbox_inches="tight")
    plt.show()


# ── call ──────────────────────────────────────────────────────────────────────
plot_2x2_ig_grid(
    base_path="data/Labelled_VEP_Data",
    model=model,
    cnn=cnn,
    device_configs=[
        {"device_folder": "PRIMA_LE_DA",  "irradiance_list": [1.09, 1.33, 1.51, 1.99]},
        {"device_folder": "PRIMA_RCS_DA", "irradiance_list": [0.29, 0.92, 1.51, 2.16]},
        {"device_folder": "MP20_LE_DA",   "irradiance_list": [0.1,  0.14, 0.41, 0.96]},
        {"device_folder": "MP20_RCS_LA",  "irradiance_list": [0.14, 0.25, 0.39, 0.59]},
    ],
    show_original=False,
    crop_dwt_start_pct=0.0,
    annotate_first_panel=True,    # ← flip to False to hide annotations
    save_path="outputs/figures/IG_example"
)